In [0]:
from pyspark.sql.functions import current_timestamp, col, lit, to_timestamp, date_format, from_utc_timestamp, input_file_name
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("year", "None", "Year (YYYY)")
dbutils.widgets.text("month", "None", "Month (MM)")

year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")

batch_processing = f"{year}{str(month).zfill(2)}"

batch_processing

In [0]:
source_path = f"/Volumes/oftalmo_sus/00_landing/raw_files/SIH/SIH_GO_{year}{month}.parquet"
df_landing = spark.read.parquet(source_path)

In [0]:
df_bronze = (
    df_landing
    .withColumn("bronze_ingest_date", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("batch_processing", lit(batch_processing))
)

In [0]:
target_table = "oftalmo_sus.01_bronze.tb_datasus_sih"

In [0]:
try:
    # Limpando os nomes das colunas
    df_bronze = df_bronze.toDF(*[col.strip() for col in df_bronze.columns])

    replace_condition = f"batch_processing = '{batch_processing}'"

    (
        df_bronze.write
        .format("delta")
        .partitionBy("batch_processing") # Particionando pelo lote do arquivo
        .mode("append")
        .option("replaceWhere", replace_condition) # Protegendo registros existentes
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"✅ Sucesso! Dados hospitalares integrados na Bronze Delta em: {target_table}")

except Exception as e:
    print(f"❌ Erro ao processar camada Bronze do SIH: {e}")
    raise e